# 02 — GRPO Reinforcement Learning

Trains GRPO Model

## 1. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Install dependencies
!pip install "protobuf<5" --quiet
!pip install unsloth --quiet
!pip install -e . --quiet
!pip install bitsandbytes latex2sympy2 --quiet
!pip install mergekit --quiet

# mergekit pulls in llm_blender which is broken with transformers>=4.45
# (TRANSFORMERS_CACHE removed). We don't use it, so uninstall.
!pip uninstall llm-blender -y --quiet 2>/dev/null; true

## 2. Config

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
#import importlib.util

from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

cfg = load_config()


setup_hf_token()
setup_wandb(cfg.grpo_training.report_to and "math-rl-grpo")

mount_google_drive()

SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"
GRPO_CHECKPOINT = None 

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Mounted at /content/drive


## 3. Run GRPO Training

In [ ]:
# Delete old merged model cache (important after SFT config changes).
# merge_adapter() skips the merge if the output directory already exists —
# so if we changed LoRA config or re-ran SFT, we must clear this cache
# to force a fresh merge with the correct weights.
!rm -rf outputs/sft_merged

from math_rl_tuning.grpo_trainer import run_grpo_training

#runs GRPO, returns trainer, model, tokenizer, and reward callback for evaluation and inference after training
trainer, model, tokenizer, reward_callback = run_grpo_training(
    cfg,
    sft_adapter_path=SFT_ADAPTER_PATH,
    save_to_drive=True,
    checkpoint_path=GRPO_CHECKPOINT,
)

## 4. Eval


In [ ]:
#shows reward history from GRPO training
reward_callback.plot()

In [ ]:
#example of grpo model after training

from math_rl_tuning.inference import generate

questions = [
    "What is 15% of 240?",
    "Solve for x: 3x + 7 = 22",
    "A rectangle has length 12 cm and width 5 cm. What is its area?",
]

for q in questions:
    print(f"Q: {q}")
    response = generate(q, model, tokenizer)
    print(f"A: {response[:300]}")
    print("-" * 40)

Q: What is 15% of 240?
A: To find 15% of 240, we follow these steps:

Step 1: Convert the percentage to a decimal. 
\[15\% = \frac{15}{100} = 0.15\]

Step 2: Multiply the decimal by the given number.
\[0.15 \times 240 = 36\]

Therefore, 15% of 240 is $\boxed{36}$.
----------------------------------------
Q: Solve for x: 3x + 7 = 22
A: To solve the equation $3x + 7 = 22$, we follow these steps:

1. Subtract 7 from both sides of the equation to isolate the term with the variable:
\[3x + 7 - 7 = 22 - 7\]
This simplifies to:
\[3x = 15\]

2. Divide both sides by 3 to solve for $x$:
\[\frac{3x}{3} = \frac{15}{3}\]
This simplifies to:
\
----------------------------------------
Q: A rectangle has length 12 cm and width 5 cm. What is its area?
A: To find the area of a rectangle, we use the formula:

\[ \text{Area} = \text{Length} \times \text{Width} \]

Given that the length of the rectangle is 12 cm and the width is 5 cm, we substitute these values into the formula:

\[ \text{Area} = 11 \, \te

## 5. Cleanup

In [8]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()

Memory cleared.
